# Guanajuato rainfed test: maize and faba bean

This focused notebook runs WOFOST 8.1 water-limited production with the classic water balance (`Wofost81_WLP_CWB`) for **maize** (`maize`) and **grain faba bean** (`fababean`) in 2024 and 2025. Both runs use `water_mode="rainfed"`; no irrigation events are supplied. The crop varieties and sowing windows below are scenario choices, not observed management or a crop-suitability recommendation.

`DATA_MODE="synthetic"` is a runnable wiring check with invented weather and soil. It does **not** validate Guanajuato yields. For `DATA_MODE="gcs"`, the weather comes from the immutable Guanajuato farm-profile run. Before using GCS mode, enter measured or independently justified soil hydraulic parameters and initial available water. The reference point is 20.4243195, -101.1466639; it is not a surveyed plot. The current profile does not contain the required soil measurements.

The optional SIAP section runs only in GCS mode. It reports municipal errors for crop/year pairs with observations. The 2024–2025 Valle de Santiago SIAP files have `Maíz grano` but no `Haba grano` rows, so faba bean has no municipal error for these years. A missing SIAP row is never replaced with an invented value. SIAP's municipal production per sown hectare and WOFOST's point-level dry-matter yield can also differ in moisture basis and spatial scale.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from pcse.input import YAMLCropDataProvider
from pcse.models import Wofost81_WLP_CWB

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / "gradientdescentfarmers" / "model_builder").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook within the repository or install the package")
sys.path.insert(0, str(repo_root))

from gradientdescentfarmers.model_builder.gcs import GCSDataClient
from gradientdescentfarmers.model_builder.wofost import available_crop_varieties, run_wofost
from gradientdescentfarmers.model_builder.calibration import compare_siap_yield, error_metrics, prepare_siap

In [ ]:
DATA_MODE = "synthetic"  # synthetic or gcs
RUN_URI = "gs://data_dump_samples/farm_profiles/guanajuato_valle_demo/runs/67b66942681a4214"
GCP_PROJECT = "gdfarming"
YEARS = (2024, 2025)
CROPS = {
    "maize": {"variety": "Grain_maize_202", "sow": "05-15", "end": "12-15"},
    "fababean": {"variety": "Faba_bean_801", "sow": "06-15", "end": "12-15"},
}
CO2_PPM = 420.0  # scenario assumption
ELEVATION_OVERRIDE_M = None
MUNICIPALITY = "Valle de Santiago"
SIAP_CROP_LABELS = {"maize": "Maíz grano", "fababean": "Haba grano"}

# Fill these before switching to GCS mode. Fractions: SMW, SMFCF, SM0, CRAIRC.
# RDMSOL and initial available water: cm. SOPE, KSUB, K0: cm/day.
SOIL_PARAMETERS = {"SMW": None, "SMFCF": None, "SM0": None, "RDMSOL": None,
                   "CRAIRC": None, "SOPE": None, "KSUB": None, "K0": None}
INITIAL_AVAILABLE_WATER_CM = None

catalog = available_crop_varieties(YAMLCropDataProvider(model=Wofost81_WLP_CWB))
for crop, selection in CROPS.items():
    assert crop in catalog, f"{crop} is not in the WOFOST 8.1 catalog"
    assert selection["variety"] in catalog[crop], f"Choose a catalog variety for {crop}: {catalog[crop]}"
    print(crop, "selected:", selection["variety"], "available:", catalog[crop])

In [ ]:
if DATA_MODE == "synthetic":
    # Invented values for a wiring test only; never use as Guanajuato observations.
    dates = pd.date_range("2024-01-01", "2025-12-31")
    weather = pd.DataFrame({"date": dates, "tmin_c": 12.0, "tmax_c": 27.0,
                            "precip_mm": 2.0, "rh_pct": 65.0, "wind_m_s": 2.0,
                            "solar_mj_m2_day": 19.0})
    soil = {"SMW": 0.1, "SMFCF": 0.3, "SM0": 0.4, "RDMSOL": 100.0,
            "CRAIRC": 0.06, "SOPE": 10.0, "KSUB": 10.0, "K0": 10.0}
    initial_water_cm = 10.0
    site = {"latitude": 20.4243195, "longitude": -101.1466639}
    elevation_m = 1700.0
    run_id = "SYNTHETIC_SMOKE"
elif DATA_MODE == "gcs":
    missing_soil = [name for name, value in SOIL_PARAMETERS.items() if value is None]
    if missing_soil or INITIAL_AVAILABLE_WATER_CM is None:
        raise ValueError(f"Supply measured soil inputs before using GCS mode: {missing_soil}; initial water is required")
    gcs = GCSDataClient(project=GCP_PROJECT)
    profile = gcs.load_profile(RUN_URI)
    weather = profile.table("weather_daily")
    site = profile.manifest["farmer_inputs"]
    assert abs(site["latitude"] - 20.4243195) < 0.01
    assert abs(site["longitude"] + 101.1466639) < 0.01
    elevation_m = ELEVATION_OVERRIDE_M if ELEVATION_OVERRIDE_M is not None else profile.manifest.get("topography", {}).get("elevation_m")
    if elevation_m is None:
        raise ValueError("Supply ELEVATION_OVERRIDE_M when profile topography has no elevation")
    soil = SOIL_PARAMETERS
    initial_water_cm = INITIAL_AVAILABLE_WATER_CM
    run_id = profile.manifest["run_id"]
else:
    raise ValueError("DATA_MODE must be synthetic or gcs")
print("Input source:", DATA_MODE, "run:", run_id, "weather rows:", len(weather))

In [ ]:
outputs = []
for year in YEARS:
    for crop, selection in CROPS.items():
        outputs.append(run_wofost(
            weather, crop=crop, variety=selection["variety"],
            sowing_date=f"{year}-{selection['sow']}",
            end_date=f"{year}-{selection['end']}",
            latitude=float(site["latitude"]), longitude=float(site["longitude"]),
            elevation_m=float(elevation_m), co2_ppm=CO2_PPM, run_id=run_id,
            water_mode="rainfed", soil_parameters=soil,
            initial_available_water_cm=initial_water_cm,
        ))
simulation_df = pd.concat(outputs, ignore_index=True)
simulation_df["input_source"] = DATA_MODE
final_df = simulation_df.loc[simulation_df.is_final_day].copy()
assert len(final_df) == len(YEARS) * len(CROPS)
assert set(final_df.water_mode) == {"rainfed"}
assert final_df.irrigation_applied_mm.eq(0).all()
assert final_df.yield_t_ha.astype(float).ge(0).all()
display(final_df[["year", "crop", "variety", "water_mode", "yield_t_ha", "input_source"]])

output_dir = repo_root / "data" / "model_builder"
output_dir.mkdir(parents=True, exist_ok=True)
simulation_csv = output_dir / f"guanajuato_maize_fababean_rainfed_{DATA_MODE}.csv"
simulation_df.to_csv(simulation_csv, index=False)
print("Saved:", simulation_csv)

## SIAP coverage and errors

GCS mode loads the 2024 and 2025 municipal SIAP files. The coverage table shows which crop/year pairs have a matching municipal observation. Error metrics are calculated only for matched pairs. Synthetic mode skips SIAP entirely because invented weather and soil cannot establish a real validation score.

In [ ]:
if DATA_MODE == "synthetic":
    print("SIAP comparison skipped: model inputs are synthetic.")
else:
    siap_prefix = "gs://data_dump_samples/SIAP_Agricultural_Production_2000_2025/"
    siap_uris = [
        f"{siap_prefix}Municipal (2003_2025)/Cierre_agricola_mun_{year}.csv"
        for year in YEARS
    ]
    siap_df = pd.concat([gcs.read_frame(uri) for uri in siap_uris], ignore_index=True)
    observed_df = prepare_siap(siap_df, municipality=MUNICIPALITY,
                               crop_labels=SIAP_CROP_LABELS)
    coverage_df = final_df[["year", "crop"]].merge(
        observed_df[["year", "crop"]].assign(has_siap=True),
        on=["year", "crop"], how="left", validate="one_to_one",
    )
    coverage_df["has_siap"] = coverage_df.has_siap.fillna(False).astype(bool)
    display(coverage_df)
    observed_pairs = pd.MultiIndex.from_frame(observed_df[["year", "crop"]])
    simulation_pairs = pd.MultiIndex.from_frame(simulation_df[["year", "crop"]])
    matched_simulation_df = simulation_df.loc[simulation_pairs.isin(observed_pairs)]
    if matched_simulation_df.empty:
        print("No SIAP comparison is available for these crop/year pairs.")
    else:
        comparison_df = compare_siap_yield(
            matched_simulation_df, siap_df, municipality=MUNICIPALITY,
            crop_labels=SIAP_CROP_LABELS,
        )
        metrics_df = pd.concat([
            error_metrics(group, prediction_column="simulated_t_ha").assign(crop=crop)
            for crop, group in comparison_df.groupby("crop")
        ], ignore_index=True)
        display(comparison_df[["year", "crop", "observed_t_ha", "simulated_t_ha",
                               "raw_error_t_ha", "raw_absolute_error_t_ha",
                               "raw_absolute_percentage_error"]])
        display(metrics_df[["crop", "n_years", "mae_t_ha", "rmse_t_ha", "mean_bias_t_ha", "mape_pct"]])
        comparison_csv = output_dir / "siap_maize_fababean_rainfed_comparison.csv"
        comparison_df.to_csv(comparison_csv, index=False)
        print("Saved:", comparison_csv)